# Shoreline Linear Regression

### Import Libraries

In [29]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
import joblib
import os

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Load Dataset

In [30]:
ROOT_DIR = Path.cwd().parent

PROCESSED_DIR = ROOT_DIR / "processed"

df = pd.read_csv(PROCESSED_DIR / "final_modeling_dataset.csv")

# Inspect the dataset
print("Dataset shape:", df.shape)
display(df.head())

# Check missing values
print(df.isnull().sum())

Dataset shape: (62323, 17)


,transect_id,winter,first_date,last_date,first_position,last_position,observation_count,shoreline_change_m,span_days,BeachWidth_m,beach_slope,shore_orientation_sin,shore_orientation_cos,winter_wave_power,wave_dir_sin,wave_dir_cos,RONI
0,usa_CA_0001-0000,2001-2002,2001-12-07 18:11:06+00:00,2002-02-25 18:11:29+00:00,253.024881,242.448636,5,-10.576246,80,112.1,0.055,-0.989016,-0.147809,5.956667,-0.986087,-0.166194,-0.2
1,usa_CA_0001-0000,2002-2003,2002-12-02 17:54:05+00:00,2003-02-20 17:55:43+00:00,258.493276,237.103652,7,-21.389624,80,112.1,0.055,-0.989016,-0.147809,5.956667,-0.986087,-0.166194,0.9
2,usa_CA_0001-0000,2003-2004,2003-12-21 18:01:13+00:00,2004-02-15 18:11:39+00:00,257.541207,245.183824,7,-12.357383,56,112.1,0.055,-0.989016,-0.147809,5.956667,-0.986087,-0.166194,0.2
3,usa_CA_0001-0000,2004-2005,2004-12-15 18:11:45+00:00,2005-02-09 18:09:00+00:00,257.237016,248.957358,5,-8.279657,55,112.1,0.055,-0.989016,-0.147809,5.956667,-0.986087,-0.166194,0.5
4,usa_CA_0001-0000,2006-2007,2006-12-13 18:17:12+00:00,2007-02-15 18:17:32+00:00,262.013544,242.402391,6,-19.611152,64,112.1,0.055,-0.989016,-0.147809,5.956667,-0.986087,-0.166194,0.6


transect_id                0
winter                     0
first_date                 0
last_date                  0
first_position             0
last_position              0
observation_count          0
shoreline_change_m         0
span_days                  0
BeachWidth_m               0
beach_slope              745
shore_orientation_sin      0
shore_orientation_cos      0
winter_wave_power          0
wave_dir_sin               0
wave_dir_cos               0
RONI                       0
dtype: int64


### Define Target Variables and Features

In [31]:
# Target variable
y = df["shoreline_change_m"]

# Model 1: Beach characteristics
features_1 = [
    "BeachWidth_m",
    "beach_slope",
    "shore_orientation_sin",
    "shore_orientation_cos", 
]

# Model 2: Beach characteristics + wave conditions
features_2 = features_1 + [
    "winter_wave_power",
    "wave_dir_sin",
    "wave_dir_cos"
]

# Model 3: Beach characteristics + wave conditions + El Niño
features_3 = features_2 + [
    "RONI"
]

### Split Training and Testing Data

In [32]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["transect_id"])
)

# Split target variable
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))

Training samples: 49849
Testing samples: 12474


### Create Training and Test Features

In [33]:
# Model 1
X1_train = df[features_1].iloc[train_idx]
X1_test = df[features_1].iloc[test_idx]

# Model 2
X2_train = df[features_2].iloc[train_idx]
X2_test = df[features_2].iloc[test_idx]

# Model 3
X3_train = df[features_3].iloc[train_idx]
X3_test = df[features_3].iloc[test_idx]

### Handle Missing Values

In [34]:
from sklearn.impute import SimpleImputer

# Create imputers
imputer_1 = SimpleImputer(strategy="median")
imputer_2 = SimpleImputer(strategy="median")
imputer_3 = SimpleImputer(strategy="median")

# Model 1
X1_train = imputer_1.fit_transform(X1_train)
X1_test = imputer_1.transform(X1_test)

# Model 2
X2_train = imputer_2.fit_transform(X2_train)
X2_test = imputer_2.transform(X2_test)

# Model 3
X3_train = imputer_3.fit_transform(X3_train)
X3_test = imputer_3.transform(X3_test)

### Train the Models

In [35]:
from sklearn.linear_model import LinearRegression

# Initialize models
model_1 = LinearRegression()
model_2 = LinearRegression()
model_3 = LinearRegression()

# Train models
model_1.fit(X1_train, y_train)
model_2.fit(X2_train, y_train)
model_3.fit(X3_train, y_train)

print("All three models trained successfully!")

All three models trained successfully!


### Save Trained Models

In [36]:
# Create models directory
os.makedirs("../models", exist_ok=True)

# Save trained models
joblib.dump(model_1, "../models/linear_regression_model_1.pkl")
joblib.dump(model_2, "../models/linear_regression_model_2.pkl")
joblib.dump(model_3, "../models/linear_regression_model_3.pkl")

print("All three models successfully saved!")

All three models successfully saved!


### Evaluate and Compare Models

In [37]:
# Generate predictions
y_pred_1 = model_1.predict(X1_test)
y_pred_2 = model_2.predict(X2_test)
y_pred_3 = model_3.predict(X3_test)

print("Predictions generated successfully!")

Predictions generated successfully!


In [38]:
# Calculate evaluation metrics
results = {
    "Model": ["Model 1", "Model 2", "Model 3"],

    "R²": [
        r2_score(y_test, y_pred_1),
        r2_score(y_test, y_pred_2),
        r2_score(y_test, y_pred_3)
    ],

    "MAE": [
        mean_absolute_error(y_test, y_pred_1),
        mean_absolute_error(y_test, y_pred_2),
        mean_absolute_error(y_test, y_pred_3)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred_1)),
        np.sqrt(mean_squared_error(y_test, y_pred_2)),
        np.sqrt(mean_squared_error(y_test, y_pred_3))
    ]
}

results_df = pd.DataFrame(results)

display(results_df.round(4))

,Model,R²,MAE,RMSE
0,Model 1,0.0042,9.9504,13.1521
1,Model 2,0.0067,9.9382,13.1354
2,Model 3,0.0184,9.8574,13.0577
